# Training Data Inspection

Inspect raw HDF5 data and the processed data that gets fed into the model.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys, os

DATA_PATH = 'japan_overfit.hdf5'
CONFIG_PATH = 'pga_configs/transformer_japan_overfit.json'

with open(CONFIG_PATH) as f:
    config = json.load(f)
print(json.dumps(config['training_params']['generator_params'][0], indent=2))

## 1. Raw HDF5 Structure

In [ ]:
def print_h5_structure(name, obj):
    indent = '  ' * name.count('/')
    if isinstance(obj, h5py.Dataset):
        print(f'{indent}{name}: shape={obj.shape}, dtype={obj.dtype}')
    else:
        print(f'{indent}{name}/')

with h5py.File(DATA_PATH, 'r') as f:
    f.visititems(print_h5_structure)

In [ ]:
# Metadata overview
with h5py.File(DATA_PATH, 'r') as f:
    print('=== Global Metadata ===')
    for key in f['metadata']:
        if key == 'event_metadata':
            continue
        val = f['metadata'][key][()]
        print(f'  {key}: {val}')
    
    print(f'\n=== Event Metadata ===')
    em = f['metadata/event_metadata']
    if isinstance(em, h5py.Group):
        cols = {}
        for col_name in em.keys():
            vals = em[col_name][()]
            if vals.dtype.kind == 'S':
                vals = np.array([v.decode() for v in vals])
            cols[col_name] = vals
        event_meta_df = pd.DataFrame(cols)
    else:
        event_meta_df = pd.read_hdf(DATA_PATH, 'metadata/event_metadata')
    
    print(f'  Columns: {list(event_meta_df.columns)}')
    print(f'  Number of events: {len(event_meta_df)}')
    display(event_meta_df.head(10))

In [ ]:
# Per-event summary: stations, waveform shape, labels
event_key = None
for k in ['KiK_File', '#EventID', 'EVENT']:
    if k in event_meta_df.columns:
        event_key = k
        break
print(f'Event key: {event_key}')

# Detect magnitude key
mag_key = config['training_params']['generator_params'][0].get('key', 'Magnitude')
pga_key = config['training_params']['generator_params'][0].get('pga_key', 'pga')
print(f'Magnitude key: {mag_key}, PGA key: {pga_key}')

event_summary = []
with h5py.File(DATA_PATH, 'r') as f:
    for _, event in event_meta_df.iterrows():
        ename = str(event[event_key])
        if ename not in f['data']:
            continue
        g = f['data'][ename]
        info = {'event': ename}
        if mag_key in event:
            info['magnitude'] = event[mag_key]
        for dkey in g:
            ds = g[dkey]
            info[f'{dkey}_shape'] = ds.shape
            if dkey == 'waveforms':
                info['n_stations'] = ds.shape[0]
                info['n_samples'] = ds.shape[1]
                info['n_channels'] = ds.shape[2]
        event_summary.append(info)

event_df = pd.DataFrame(event_summary)
print(f'\nTotal events in HDF5: {len(event_df)}')
print(f'Stations per event: min={event_df["n_stations"].min()}, max={event_df["n_stations"].max()}, mean={event_df["n_stations"].mean():.1f}')
if 'magnitude' in event_df.columns:
    print(f'Magnitude range: [{event_df["magnitude"].min():.2f}, {event_df["magnitude"].max():.2f}]')
display(event_df.head(20))

## 2. Label Distributions

In [ ]:
# Magnitude distribution
if 'magnitude' in event_df.columns:
    fig, ax = plt.subplots(1, 1, figsize=(8, 3))
    ax.hist(event_df['magnitude'], bins=30, edgecolor='black')
    ax.set_xlabel('Magnitude')
    ax.set_ylabel('Count')
    ax.set_title('Magnitude Distribution')
    plt.tight_layout()
    plt.show()

In [ ]:
# Location distribution
from gemini_util_light import detect_location_keys
coord_keys = detect_location_keys(event_meta_df.columns)
print(f'Location keys: {coord_keys}')

if len(coord_keys) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(event_meta_df[coord_keys[1]], event_meta_df[coord_keys[0]], 
                    c=event_meta_df.get(mag_key, None), cmap='viridis', s=20)
    axes[0].set_xlabel(coord_keys[1])
    axes[0].set_ylabel(coord_keys[0])
    axes[0].set_title('Event Locations (color=magnitude)')
    
    for i, key in enumerate(coord_keys):
        axes[i+1 if i < 2 else 2].hist(event_meta_df[key], bins=30, edgecolor='black')
        axes[i+1 if i < 2 else 2].set_title(key)
    plt.tight_layout()
    plt.show()

In [ ]:
# PGA distribution (across all events)
all_pga = []
with h5py.File(DATA_PATH, 'r') as f:
    for _, event in event_meta_df.iterrows():
        ename = str(event[event_key])
        if ename not in f['data']:
            continue
        g = f['data'][ename]
        if pga_key in g:
            all_pga.append(g[pga_key][()])

if all_pga:
    all_pga_flat = np.concatenate(all_pga)
    print(f'PGA values: n={len(all_pga_flat)}, range=[{all_pga_flat.min():.4f}, {all_pga_flat.max():.4f}], mean={all_pga_flat.mean():.4f}')
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].hist(all_pga_flat, bins=50, edgecolor='black')
    axes[0].set_title('PGA Distribution')
    axes[0].set_xlabel('PGA')
    axes[1].hist(all_pga_flat[all_pga_flat > 0], bins=50, edgecolor='black')
    axes[1].set_title('PGA Distribution (nonzero only)')
    axes[1].set_xlabel('PGA')
    plt.tight_layout()
    plt.show()
else:
    print('No PGA data found')

## 3. Raw Waveforms

In [ ]:
# Plot waveforms for a few events
N_EVENTS = min(3, len(event_meta_df))
N_STATIONS = 5  # stations to show per event
CHANNEL_NAMES = ['E', 'N', 'Z']

with h5py.File(DATA_PATH, 'r') as f:
    for ev_idx in range(N_EVENTS):
        event = event_meta_df.iloc[ev_idx]
        ename = str(event[event_key])
        if ename not in f['data']:
            continue
        g = f['data'][ename]
        waveforms = g['waveforms'][()]  # (n_stations, n_samples, 3)
        p_picks = g['p_picks'][()] if 'p_picks' in g else None
        pga_vals = g[pga_key][()] if pga_key in g else None
        coords = g['coords'][()] if 'coords' in g else None
        
        n_sta = min(N_STATIONS, waveforms.shape[0])
        mag_val = event.get(mag_key, '?')
        
        fig, axes = plt.subplots(n_sta, 3, figsize=(16, 2.5 * n_sta), sharex=True)
        if n_sta == 1:
            axes = axes[np.newaxis, :]
        fig.suptitle(f'Event {ename} | Mag={mag_val} | {waveforms.shape[0]} stations', fontsize=13)
        
        for s in range(n_sta):
            for c in range(3):
                ax = axes[s, c]
                ax.plot(waveforms[s, :, c], linewidth=0.5)
                if p_picks is not None and p_picks[s] > 0:
                    ax.axvline(p_picks[s], color='r', linewidth=0.8, label=f'P={int(p_picks[s])}')
                if s == 0:
                    ax.set_title(CHANNEL_NAMES[c])
                if c == 0:
                    sta_label = f'Sta {s}'
                    if pga_vals is not None:
                        sta_label += f' PGA={pga_vals[s]:.3f}'
                    if coords is not None:
                        sta_label += f'\n({coords[s,0]:.2f}, {coords[s,1]:.2f})'
                    ax.set_ylabel(sta_label, fontsize=8)
                if p_picks is not None and p_picks[s] > 0 and s == 0 and c == 0:
                    ax.legend(fontsize=7)
        plt.tight_layout()
        plt.show()

## 4. Station Coordinates & P-picks

In [ ]:
# Show station distribution and P-pick statistics for first few events
with h5py.File(DATA_PATH, 'r') as f:
    for ev_idx in range(min(3, len(event_meta_df))):
        event = event_meta_df.iloc[ev_idx]
        ename = str(event[event_key])
        if ename not in f['data']:
            continue
        g = f['data'][ename]
        coords = g['coords'][()] if 'coords' in g else None
        p_picks = g['p_picks'][()] if 'p_picks' in g else None
        pga_vals = g[pga_key][()] if pga_key in g else None
        
        if coords is None:
            continue
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f'Event {ename} | {coords.shape[0]} stations')
        
        # Station map
        sc = axes[0].scatter(coords[:, 1], coords[:, 0], c=pga_vals if pga_vals is not None else 'blue',
                            cmap='hot_r', s=40, edgecolors='black', linewidth=0.5)
        if coord_keys and len(coord_keys) >= 2:
            ev_loc = event.get(coord_keys[0], None), event.get(coord_keys[1], None)
            if ev_loc[0] is not None:
                axes[0].plot(ev_loc[1], ev_loc[0], '*', color='red', markersize=15, label='Epicenter')
                axes[0].legend()
        axes[0].set_xlabel('Lon')
        axes[0].set_ylabel('Lat')
        axes[0].set_title('Station Map (color=PGA)')
        plt.colorbar(sc, ax=axes[0])
        
        # P-pick histogram
        if p_picks is not None:
            valid_picks = p_picks[p_picks > 0]
            axes[1].hist(valid_picks, bins=30, edgecolor='black')
            axes[1].set_title(f'P-pick times ({len(valid_picks)}/{len(p_picks)} valid)')
            axes[1].set_xlabel('Sample index')
        
        # PGA vs distance
        if pga_vals is not None and coord_keys and len(coord_keys) >= 2:
            ev_lat, ev_lon = event.get(coord_keys[0], 0), event.get(coord_keys[1], 0)
            dist = np.sqrt((coords[:, 0] - ev_lat)**2 + (coords[:, 1] - ev_lon)**2)
            axes[2].scatter(dist, pga_vals, s=20)
            axes[2].set_xlabel('Approx. distance (deg)')
            axes[2].set_ylabel('PGA')
            axes[2].set_title('PGA vs Distance')
        
        plt.tight_layout()
        plt.show()

## 5. Processed Data (as fed into model)

Instantiate the training data pipeline and inspect what the model actually receives.

In [ ]:
sys.path.insert(0, '.')
sys.path.insert(0, './diting')
import loader_light as loader
import gemini_util_light as util

training_params = config['training_params']
generator_params = training_params['generator_params']
if isinstance(training_params['data_path'], str):
    training_params['data_path'] = [training_params['data_path']]

data_path = training_params['data_path'][0]
gen_param = generator_params[0]

In [ ]:
# Load events (train split)
full_data = loader.load_events(
    data_path, 
    event_metadata_path='_inspect_ev.csv',
    parts=(True, True, False),  # train + dev
    shuffle_train_dev=gen_param.get('shuffle_train_dev', False),
    custom_split=gen_param.get('custom_split', None),
)
event_metadata, metadata_dict, data_loaded = full_data

print(f'Event metadata shape: {event_metadata.shape}')
print(f'Event metadata columns: {list(event_metadata.columns)}')
display(event_metadata.head(10))

# Count unique events
for k in ['KiK_File', '#EventID', 'EVENT']:
    if k in event_metadata.columns:
        n_unique = event_metadata[k].nunique()
        print(f'\nUnique events ({k}): {n_unique}')
        break

print(f'\nMetadata dict keys: {list(metadata_dict.keys())}')
for k, v in metadata_dict.items():
    if isinstance(v, np.ndarray):
        print(f'  {k}: shape={v.shape}, dtype={v.dtype}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# Subset to first 10 events (matching --overfit_n 10)
OVERFIT_N = 10
for k in ['KiK_File', '#EventID', 'EVENT']:
    if k in event_metadata.columns:
        ek = k
        break
unique_events = event_metadata[ek].unique()[:OVERFIT_N]
event_metadata_sub = event_metadata[event_metadata[ek].isin(unique_events)].copy()
print(f'Subset: {OVERFIT_N} events, {len(event_metadata_sub)} station-rows')
print(f'Events: {unique_events}')
if mag_key in event_metadata_sub.columns:
    mags = event_metadata_sub.drop_duplicates(subset=ek)[mag_key]
    print(f'Magnitudes: {mags.values}')

In [ ]:
# Create PreloadedEventGenerator (same as training)
sampling_rate = metadata_dict.get('sampling_rate', 100)
noise_seconds = gen_param.get('noise_seconds', 5)
cutout_start = gen_param.get('cutout_start', -1)
cutout_end = gen_param.get('cutout_end', 90)
cutout = (int(sampling_rate * (noise_seconds + cutout_start)),
          int(sampling_rate * (noise_seconds + cutout_end)))
print(f'Sampling rate: {sampling_rate}')
print(f'Cutout range (samples): {cutout}')

n_pga_targets = config['model_params'].get('n_pga_targets', 0)
max_stations = config['model_params']['max_stations']
pos_offset = gen_param.get('pos_offset', (37, 140))

generator = util.PreloadedEventGenerator(
    event_metadata_sub,
    metadata_dict,
    data_path,
    gen_param,
    cutout=cutout,
    key=gen_param.get('key', 'Magnitude'),
    label_smoothing=False,
    max_stations=max_stations,
    pga_targets=n_pga_targets,
    pos_offset=pos_offset,
    oversample=1,
    shuffle=False,
    select_first=True,
    pga_key=gen_param.get('pga_key', 'pga'),
    scale_metadata=gen_param.get('scale_metadata', False),
    trigger_based=gen_param.get('trigger_based', False),
    disable_station_foreshadowing=gen_param.get('disable_station_foreshadowing', False),
    pga_from_inactive=gen_param.get('pga_from_inactive', True),
    no_event_token=config['model_params'].get('no_event_token', False),
    selection_skew=gen_param.get('selection_skew', None),
    pga_selection_skew=gen_param.get('pga_selection_skew', None),
    magnitude_resampling=gen_param.get('magnitude_resampling', 1.0),
    min_upsample_magnitude=gen_param.get('min_upsample_magnitude', 8),
)
print(f'Generator length: {len(generator)} samples')

In [ ]:
# Inspect a few samples from the generator
N_SAMPLES = min(5, len(generator))

for i in range(N_SAMPLES):
    inputs, labels, p_picks = generator[i]
    print(f'\n=== Sample {i} ===')
    print(f'Inputs: {len(inputs)} tensors')
    for j, inp in enumerate(inputs):
        print(f'  input[{j}]: shape={inp.shape}, dtype={inp.dtype}')
    print(f'Labels: {len(labels)} tensors')
    for j, lab in enumerate(labels):
        if hasattr(lab, 'shape'):
            print(f'  label[{j}]: shape={lab.shape}, min={lab.min():.4f}, max={lab.max():.4f}')
        else:
            print(f'  label[{j}]: {lab}')
    print(f'P-picks: {p_picks}')

In [ ]:
# Decode model inputs for sample 0
inputs, labels, p_picks_info = generator[0]
waveform_inp = inputs[0].numpy()   # (1, max_stations, n_samples, 3)
metadata_inp = inputs[1].numpy()   # (1, max_stations+, 3)

print(f'Waveform input: {waveform_inp.shape}')
print(f'Metadata input: {metadata_inp.shape}')

# Active stations (nonzero waveforms)
active = np.any(waveform_inp[0] != 0, axis=(1, 2))
print(f'Active stations: {active.sum()} / {waveform_inp.shape[1]}')

# Nonzero fraction per active station
for s in range(min(5, int(active.sum()))):
    w = waveform_inp[0, s]
    nonzero_frac = np.count_nonzero(w) / w.size
    amp = np.max(np.abs(w))
    print(f'  Station {s}: nonzero={nonzero_frac:.2%}, max_amp={amp:.4f}')

if len(inputs) > 2:
    pga_targets_inp = inputs[2].numpy()  # (1, n_pga_targets, 3)
    print(f'\nPGA targets input: {pga_targets_inp.shape}')

# Labels
print(f'\nMagnitude label: {labels[0]}')
print(f'Location label: {labels[1]}')
if len(labels) > 2:
    print(f'PGA label: shape={labels[2].shape}, values={labels[2].flatten()[:10]}')

In [ ]:
# Plot processed waveforms (as model sees them, after cutout + right-alignment)
inputs, labels, _ = generator[0]
waveform_inp = inputs[0].numpy()[0]  # (max_stations, n_samples, 3)

active = np.any(waveform_inp != 0, axis=(1, 2))
n_active = int(active.sum())
n_show = min(6, n_active)

fig, axes = plt.subplots(n_show, 3, figsize=(16, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Processed waveforms (sample 0) | {n_active} active stations', fontsize=13)

active_idx = np.where(active)[0]
for si, s in enumerate(active_idx[:n_show]):
    for c in range(3):
        ax = axes[si, c]
        ax.plot(waveform_inp[s, :, c], linewidth=0.5)
        if si == 0:
            ax.set_title(CHANNEL_NAMES[c])
        if c == 0:
            ax.set_ylabel(f'Sta {s}', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Compare waveforms across events for the same station index
STA_IDX = 0  # first active station
n_compare = min(5, len(generator))

fig, axes = plt.subplots(n_compare, 3, figsize=(16, 2.5 * n_compare), sharex=True)
if n_compare == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Station index {STA_IDX} across {n_compare} events (processed)', fontsize=13)

for i in range(n_compare):
    inputs, labels, _ = generator[i]
    w = inputs[0].numpy()[0, STA_IDX]  # (n_samples, 3)
    mag = labels[0].flatten()[0]
    for c in range(3):
        ax = axes[i, c]
        ax.plot(w[:, c], linewidth=0.5)
        if i == 0:
            ax.set_title(CHANNEL_NAMES[c])
        if c == 0:
            ax.set_ylabel(f'Event {i}\nM={mag:.1f}', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics across all generator samples
all_mags = []
all_locs = []
all_pgas = []
all_n_active = []

for i in range(len(generator)):
    inputs, labels, _ = generator[i]
    wf = inputs[0].numpy()[0]
    active = np.any(wf != 0, axis=(1, 2))
    all_n_active.append(active.sum())
    all_mags.append(labels[0].flatten()[0])
    all_locs.append(labels[1].flatten())
    if len(labels) > 2:
        all_pgas.append(labels[2].flatten())

print(f'Total samples: {len(generator)}')
print(f'Active stations: min={min(all_n_active)}, max={max(all_n_active)}, mean={np.mean(all_n_active):.1f}')
print(f'Magnitude: min={min(all_mags):.2f}, max={max(all_mags):.2f}, unique={len(set([round(m,1) for m in all_mags]))}')
if all_locs:
    locs = np.array(all_locs)
    print(f'Location: lat=[{locs[:,0].min():.2f}, {locs[:,0].max():.2f}], '
          f'lon=[{locs[:,1].min():.2f}, {locs[:,1].max():.2f}], '
          f'depth=[{locs[:,2].min():.2f}, {locs[:,2].max():.2f}]')
if all_pgas:
    pgas = np.concatenate(all_pgas)
    nonzero_pgas = pgas[pgas != 0]
    print(f'PGA: n_total={len(pgas)}, n_nonzero={len(nonzero_pgas)}, '
          f'range=[{nonzero_pgas.min():.4f}, {nonzero_pgas.max():.4f}], mean={nonzero_pgas.mean():.4f}')

In [ ]:
# Clean up temp cache
import glob
for f in glob.glob('_inspect_ev*.csv'):
    os.remove(f)
    print(f'Removed {f}')